In [116]:
%cd /content
import os
if not os.path.exists("valley-outfitters-analysis"):
    !git clone https://github.com/LSRajan/valley-outfitters-analysis.git
%cd valley-outfitters-analysis

import pandas as pd
pd.set_option("display.max_columns", None)

/content
/content/valley-outfitters-analysis


In [117]:
orders_raw = pd.read_csv("data/raw/orders.csv")
customers_raw = pd.read_csv("data/raw/customers.csv")
order_items_raw = pd.read_csv("data/raw/order_items.csv")
products_raw = pd.read_csv("data/raw/products.csv")
returns_raw = pd.read_csv("data/raw/returns.csv")
web_sessions_raw = pd.read_csv("data/raw/web_sessions.csv")

In [118]:
def clean_labels(col, mapping, allowed):
  col = col.str.strip().str.lower().replace(mapping)
  #print(col[~col.isin(allowed) & col.notna()])
  assert len(col[~col.isin(allowed) & col.notna()]) == 0
  return col

def clean_dates(col, formats):
  result = pd.Series(pd.NaT, index=col.index, dtype="datetime64[ns]")
  for fmt in formats:
      parsed = pd.to_datetime(col, format=fmt, errors="coerce")
      result = result.fillna(parsed)
  unparsed = col[col.notna() & result.isna()]
  print(f"{col.name}: {result.notna().sum()} parsed, {len(unparsed)} left as NA")
  print("   unparsed values:", unparsed.value_counts().to_dict())
  return result

DATE_FORMATS = [
    "%Y-%m-%d",            # 2025-09-24
    "%Y-%m-%d %H:%M:%S",   # 2026-03-13 15:45:56
    "%Y/%m/%d",            # 2025/09/24
    "%d/%m/%Y",            # 30/09/2026  (slashes: day first)
    "%d.%m.%y",            # 30.09.26    (dots: day first, 2-digit year)
    "%m-%d-%Y",            # 09-30-2026  (dashes: month first)
    "%d %b %Y",            # 30 Sep 2026
    "%B %d, %Y",           # September 30, 2026
]


#1. Orders

In [119]:
orders = orders_raw.copy()

###Fix 01 : Duplicate rows in orders table are dropped

In [120]:
orders = orders.drop_duplicates().copy()

### Fix 03: Capitalise all customer ids

In [121]:
orders["customer_id"] = orders["customer_id"].str.upper()

###Fix 06: Clean order dates to the standard format

In [122]:
orders["order_date"] = clean_dates(orders["order_date"], DATE_FORMATS).dt.normalize()

order_date: 24463 parsed, 193 left as NA
   unparsed values: {'0000-00-00': 66, 'TBC': 65, 'unknown': 62}


###Fix 07: Clean ship dates to the standard format

In [123]:
orders["ship_date"] = clean_dates(orders["ship_date"], DATE_FORMATS)

ship_date: 21607 parsed, 147 left as NA
   unparsed values: {'TBC': 52, '0000-00-00': 50, 'unknown': 45}


###Fix 08: Standardise and relabel status labels

In [124]:
status_map = {
    "canceled" : "cancelled",
    "cancled" : "cancelled",
    "return" : "returned",
    "deliverd" : "delivered",
    "complete" : "delivered",
    "completed" : "delivered",
    "in progress" : "processing",
    "in transit" : "shipped"
}
status_allowed = ["delivered", "cancelled", "returned", "shipped", "pending", "processing"]

orders["status"] = clean_labels(orders["status"], status_map, status_allowed)

In [125]:
orders["payment_method"].str.strip().str.lower().value_counts()

,count
payment_method,
klarna,4933
paypal,4729
apple pay,4527
gift card,4490
card,4456
giftcard,506
applepay,501
pp,310
debit card,292


###Fix 09: Standardise and relabel payment method labels

In [126]:
payment_map = {
    "credit card" : "card",
    "debit card" : "card",
    "pp" : "paypal",
    "applepay" : "apple pay",
    "giftcard" : "gift card"
}
payment_allowed = ["card", "klarna", "apple pay", "gift card", "paypal"]

orders["payment_method"] = clean_labels(orders["payment_method"], payment_map, payment_allowed)

###Fix 10: Standardise channel labels

In [127]:
orders["channel"].str.strip().str.lower().value_counts()

,count
channel,
web,12831
app,6260
marketplace,3050
in-store,1314
online,797
phone,748


In [128]:
orders["channel"] = orders["channel"].str.lower().str.strip()
orders["channel"] = orders["channel"].replace("online", "web")

###Fix 11: Standardise discount codes

In [129]:
orders["discount_code"] = orders["discount_code"].str.strip().str.lower()

###Fix 12 & Fix 13: Remove "£" prefix from order total. Take absolute value to account for negative values.

In [130]:
orders["order_total"] = orders["order_total"].str.replace("£", "").astype(float).abs()

#2. Customers

In [131]:
customers = customers_raw.copy()

###Fix 14: Strip and lower first and last names.

In [132]:
customers["first_name"] = customers["first_name"].str.strip().str.lower()
customers["last_name"] = customers["last_name"].str.strip().str.lower()

###Fix 16: Clean sign up date

In [133]:
customers["signup_date"] = clean_dates(customers["signup_date"], DATE_FORMATS)

signup_date: 5215 parsed, 36 left as NA
   unparsed values: {'unknown': 12, 'TBC': 12, '0000-00-00': 12}


###Fix 17: Standardise date of birth format and fix impossible values

In [134]:
dob = clean_dates(customers["date_of_birth"], DATE_FORMATS)
dob = dob.mask(dob.isin(pd.to_datetime(["1900-01-01", "2031-04-12"])))   # placeholder dates -> NA
future = dob > pd.Timestamp.today()
dob = dob.mask(future, dob - pd.DateOffset(years=100))                   # fix 2-digit years
customers["date_of_birth"] = dob

date_of_birth: 5163 parsed, 44 left as NA
   unparsed values: {'0000-00-00': 17, 'unknown': 15, 'TBC': 12}


###Fix 15: Removed duplicate customer appearances in the customer table by retaining only the most recently created account per customer. The email (as well as postcode and phone) columns are also dropped as it is no longer useful.

In [135]:
customers["email"] = customers["email"].str.strip().str.lower()

#get customers who have signed up twice, identified by same email and date of birth. sort by most recent sign up date first
duplicates = customers.loc[customers[["email", "date_of_birth"]].duplicated(keep=False)
& customers["email"].notna()
& customers["date_of_birth"].notna()].sort_values(["email", "date_of_birth", "signup_date"], ascending=[True,True,False])

#get the id of the account with the most recent sign up date by group
duplicates["keep"] = duplicates.groupby(["email", "date_of_birth"])["customer_id"].transform("first")


id_map = (duplicates.loc[duplicates["customer_id"] != duplicates["keep"]]
               .set_index("customer_id")["keep"].to_dict())

customers = customers[~customers["customer_id"].isin(id_map)].copy()

In [136]:
customers.drop('email', axis=1, inplace=True)
customers.drop('postcode', axis=1, inplace=True)
customers.drop('phone', axis=1, inplace=True)

###Fix 18: Standardise and relabel city values

In [137]:
cities_map = {
    "cardif" : "cardiff",
    "londn" : "london",
    "manchestor": "manchester",
    "swanzea" : "swansea",
    "edinburg" : "edinburgh",
    "birmingam" : "birmingham",
}
cities_allowed = ["bristol","aberystwyth","sheffield","swansea","newport","glasgow","bath","edinburgh","exeter","bangor",
"nottingham","birmingham","leeds","cardiff","belfast","liverpool","wrexham","london","port talbot","manchester"]

customers["city"] = clean_labels(customers["city"], cities_map, cities_allowed)

###Fix 19: Relabel country labels

In [138]:
#country can be inferred from both city name and postcode when a city exists in multiple countries
countries_map = {
    "bristol": "england",
    "sheffield": "england",
    "bath": "england",
    "exeter": "england",
    "nottingham": "england",
    "birmingham": "england",
    "leeds": "england",
    "liverpool": "england",
    "london": "england",
    "manchester": "england",

    "aberystwyth": "wales",
    "swansea": "wales",
    "newport": "wales",
    "bangor": "wales",
    "cardiff": "wales",
    "wrexham": "wales",
    "port talbot": "wales",

    "glasgow": "scotland",
    "edinburgh": "scotland",

    "belfast": "northern ireland"
}
countries_allowed = ["england", "wales", "scotland", "northern ireland"]
customers["country"] = customers["city"].map(countries_map)
assert len(customers[~customers["country"].isin(countries_allowed) & customers["country"].notna()]) == 0

###Fix 20: Standardise and relabel marketing opt in labels

In [139]:
marketing_map = {
    "no" : False,
    "n" : False,
    "0" : False,
    "false" : False,
    "y" : True,
    "yes" : True,
    "1" : True,
    "true" : True
}
marketing_allowed = [True, False]
customers["marketing_opt_in"] = clean_labels(customers["marketing_opt_in"], marketing_map, marketing_allowed)

#3. Order Items

In [140]:
order_items = order_items_raw.copy()

###Fix 21: Drop the 250 duplicated rows

In [141]:
order_items = order_items.drop_duplicates().copy()
order_items.duplicated().sum()

np.int64(0)

##Fix 22: Capitalise product id string

In [142]:
order_items["product_id"] = order_items["product_id"].str.upper()

##Fix 24: Convert quantities to integer and take absolute value.

In [143]:
order_items["quantity"] = order_items["quantity"].replace({"one":1, "two":2, "three":3}).astype(int).abs()

##Fix 25: Replace outlier values with NA

In [144]:
order_items["quantity"] = order_items["quantity"].replace({0: pd.NA, 100: pd.NA, 250: pd.NA, 999: pd.NA}).astype("Int64")

##Fix 26: Remove "£" prefix from unit price

In [145]:
order_items["unit_price"] = order_items["unit_price"].str.replace("£", "").astype(float)

#4. Products

In [146]:
products = products_raw.copy()

##Fix 27.0: Removed duplicated row in products

In [147]:
products = products.drop_duplicates().copy()

##Fix 27.1: Stripped and lowered name values

In [148]:
products["product_name"] = products["product_name"].str.strip().str.lower()

##Fix 27.2: Standardise and relabel category names

In [149]:
category_map = {
    "foot wear" : "footwear",
    "outer wear": "outerwear",
    "baselayers" : "base layers",
    "back packs" : "backpacks",
    "accesories" : "accessories",
    "camping & hiking" : "camping"
}
category_allowed = ["camping", "outerwear", "accessories", "base layers", "footwear", "backpacks"]
products["category"] = clean_labels(products["category"], category_map, category_allowed)

##Fix 28: Remove "£" and fix "," symbol from unit price

In [150]:
products["unit_price"] = products["unit_price"].str.replace("£", "").str.replace(",", ".").astype(float)

##Fix 29: Take absolute value of stock quantity

In [151]:
products["stock_qty"] = products["stock_qty"].abs()

##Fix 30: Standardise and relabel supplier names

In [152]:
supplier_map = {
    "northern gear ltd" : "northern gear",
    "blue ridge textiles" : "blueridge textiles",
    "blueridge textiles ltd" : "blueridge textiles",
    "trekco limited" : "trekco ltd",
    "alpine supply co" : "alpine supply company",
    "alpine supply co." : "alpine supply company",
    "summit wholesale ltd" : "summit wholesale"
}
supplier_allowed = ["northern gear", "blueridge textiles", "trekco ltd", "alpine supply company", "summit wholesale"]
products["supplier"] = clean_labels(products["supplier"], supplier_map, supplier_allowed)

##FIX 31: Standardise launch date format

In [153]:
products["launch_date"] = clean_dates(products["launch_date"], DATE_FORMATS)

launch_date: 161 parsed, 0 left as NA
   unparsed values: {}


##Fix 32: Standardise and relabel is active values

In [154]:
active_map = {
    "n" : False,
    "no" : False,
    "0" : False,
    "false" : False,
    "y" : True,
    "yes" : True,
    "1" : True,
    "true" : True
}
active_allowed = [True, False]
products["is_active"] = clean_labels(products["is_active"], active_map, active_allowed)

/tmp/ipykernel_3884/686929864.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  col = col.str.strip().str.lower().replace(mapping)


#5. Returns

In [155]:
returns = returns_raw.copy()

##FIX 33: Standardise return date format

In [156]:
returns["return_date"] = clean_dates(returns["return_date"], DATE_FORMATS)

return_date: 2731 parsed, 30 left as NA
   unparsed values: {'TBC': 10, 'unknown': 10, '0000-00-00': 10}


##Fix 34: Standardise and relabel return reasons

In [157]:
reason_map = {
    "-" : pd.NA,
    "see email" : pd.NA,
    "too small" : "size",
    "didn't fit" : "size",
    "wrong size" : "size",
    "didnt fit" : "size",
    "doesn't fit" : "size",
    "too big" : "size",

    "found cheaper" : "changed mind",
    "no longer needed" : "changed mind",

    "box crushed" : "faulty delivery",
    "damaged" : "faulty delivery",
    "damaged in transit" : "faulty delivery",
    "arrived damaged" : "faulty delivery",
    "faulty zip" : "faulty product",

    "stitching came apart" : "faulty product",
    "not waterproof" : "faulty product",
    "zip broke" : "faulty product",
    "faulty" : "faulty product",
    "leaking" : "faulty product",

    "colour different" : "misleading",
    "not as described" : "misleading",
    "looks different to photo" : "misleading",

    "bought by mistake" : "mistake",

}
reason_allowed = ["size", "changed mind", "faulty delivery", "faulty product", "mistake", "misleading", "other"]
returns["reason"] = clean_labels(returns["reason"], reason_map, reason_allowed)

##Fix 35: Strip and lower resolution column

In [158]:
returns["resolution"] = returns["resolution"].str.strip().str.lower()

#6. Web Sessions

In [159]:
web_sessions = web_sessions_raw

##Fix 36.0 Drop duplicated rows

In [160]:
web_sessions = web_sessions.drop_duplicates().copy()

##Fix 37: Standardise session timestamp column.

In [161]:
TIMESTAMP_FORMATS = [
    "%Y-%m-%d %H:%M:%S",      # 2024-11-05 20:21:08
    "%d/%m/%Y %H:%M",         # 13/11/2025 04:13  (day first)
    "%Y-%m-%dT%H:%M:%SZ",     # 2025-06-17T12:02:46Z
]
web_sessions["session_timestamp"] = clean_dates(web_sessions["session_timestamp"], TIMESTAMP_FORMATS)

session_timestamp: 60000 parsed, 0 left as NA
   unparsed values: {}


##Fix 38: Standardise and relabel device labels.

In [162]:
device_map = {
    "iphone" : "mobile",
    "pc" : "desktop",
    "ipad" : "tablet"
}
device_allowed = ["mobile", "desktop", "tablet"]
web_sessions["device"] = clean_labels(web_sessions["device"], device_map, device_allowed)

##Fix 39: Standardise and relabel traffice source values.

In [163]:
traffic_map = {
    "none" : "direct",
    "e-mail" : "email",
    "(direct)" : "direct",
    "paid_search" : "paid search",
    "instagram" : "social",
    "facebook" : "social",
    "fb" : "social",
    "newsletter" : "email",
    "google organic" : "organic",
    "organic search" : "organic",
    "referral site" : "referral",
    "google_ads" : "ads",
    "google ads" : "ads",
    "ppc" : "ads"
}
traffic_allowed = ["email", "direct", "paid search", "ads", "social", "organic", "referral"]
web_sessions["traffic_source"] = clean_labels(web_sessions["traffic_source"], traffic_map, traffic_allowed)

##Fix 40: Take absolute value of pages viewed.

In [164]:
web_sessions["pages_viewed"] = web_sessions["pages_viewed"].abs()

##Fix 41: Standardise and relabel converted column values

In [165]:
convert_map = {
    "0" : False,
    "n" : False,
    "false" : False,
    "1" : True,
    "y" : True,
    "true" : True
}
convert_allowed = [True, False]
web_sessions["converted"] = clean_labels(web_sessions["converted"], convert_map, convert_allowed)

/tmp/ipykernel_3884/686929864.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  col = col.str.strip().str.lower().replace(mapping)


#7. Cross-Table

##Fix 42: Replace the incorrect unit prices in the order items table with the correct prices from the products table. For product ids in the order items table with no corresponding value in the products table, keep the original price.

In [166]:
merged_prices = order_items.merge(products, how = "left", on = "product_id")
order_items["unit_price"] = merged_prices["unit_price_y"].fillna(merged_prices["unit_price_x"]).values

##Fix 43: Customer ids were removed and the most recent customer id was kept for customers with multiple accounts. The old values in related tables need to be updated.

In [167]:
orders["customer_id"] = orders["customer_id"].replace(id_map)
web_sessions["customer_id"] = web_sessions["customer_id"].replace(id_map)

##Fix 44: Cap refund amount at unit price

In [168]:
ret_pro = returns.merge(order_items, how = "left", on = "order_item_id")
ret_pro

returns["refund_amount"] = ret_pro[["refund_amount", "unit_price"]].min(axis=1, skipna=False)

#Saving clean tables

In [170]:
orders.to_csv("orders.csv", index=False)
customers.to_csv("customers.csv", index=False)
order_items.to_csv("order_items.csv", index=False)
products.to_csv("products.csv", index=False)
returns.to_csv("returns.csv", index=False)
web_sessions.to_csv("web_sessions.csv", index=False)